# Sabana de Bogotá · fine-tuning Prithvi-EO-2.0

Run this in **Google Colab with a GPU** (*Runtime → Change runtime type → T4 GPU*).

Before starting you need, in Google Drive, the folder `sabana_lc` produced by `gee/01_export_dataset.js`
(`s2_2021_hls6.tif`, `labels_2021.tif`, and the four `ref_*.csv` / `strata_pixels.csv` files),
plus your file `ref_labels.csv` with the photo-interpreted classes (see `docs/protocolo_verificacion.md`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/sabana_lc'
WORK = '/content/drive/MyDrive/sabana_lc/work'   # on Drive: survives a disconnect
!ls -lh $DATA

In [ ]:
!git clone -q https://github.com/Kzamudioq/sabana-landcover-fm.git
%cd sabana-landcover-fm
!pip install -q -e ".[model]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Chips, normalisation statistics and Dynamic World map
`--stride 112` overlaps chips by half, which doubles the training set without crossing split borders.

In [ ]:
!python -m sabana_lc prepare --s2 $DATA/s2_2021_hls6.tif --labels $DATA/labels_2021.tif --out $WORK --stride 112

## 2. Baselines: K-Means (one model for all pixels) and Random Forest

In [ ]:
!python -m sabana_lc baselines --work $WORK

## 3. Fine-tune Prithvi-EO-2.0-300M-TL
About 25–40 min on a T4. The best checkpoint (by validation mIoU) is saved to `work/runs/prithvi/best.ckpt`.
If the GPU runs out of memory, use `--batch 4`.

In [ ]:
!python -m sabana_lc train --work $WORK --backbone prithvi_eo_v2_300_tl --epochs 40 --batch 8 --lr 1e-4

Optional ablation: same backbone **frozen** (only the decoder learns). It shows how much the pretrained
features alone are worth.

In [ ]:
# !python -m sabana_lc train --work $WORK --name prithvi_frozen --freeze --epochs 40 --batch 16 --lr 3e-4

## 4. Map the whole Sabana

In [ ]:
!python -m sabana_lc predict --work $WORK --ckpt $WORK/runs/prithvi/best.ckpt
# !python -m sabana_lc predict --work $WORK --name prithvi_frozen --ckpt $WORK/runs/prithvi_frozen/best.ckpt

## 5. Accuracy against the reference points
Needs `ref_labels.csv` (your photo-interpretation) in `$DATA`. Results go to `work/results/`.

In [ ]:
!mkdir -p $DATA/ref && cp $DATA/ref_points.csv $DATA/ref_strata.csv $DATA/strata_pixels.csv $DATA/ref_labels.csv $DATA/ref/
!python -m sabana_lc evaluate --work $WORK --ref $DATA/ref

In [ ]:
from IPython.display import Image, display
display(Image(f'{WORK}/results/f1_by_class.png'))
display(Image(f'{WORK}/results/maps_panel.png'))